In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp,"martin2008tubes")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "martinordas2008tubes_MATRIX_NEW VERSION TRAP TUBE TRAP TABLE_11_01_07.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
conversion = os.path.join(original_data_pathway, 'martinordas2008tubes_MATRIX_NEW VERSION TRAP TUBE TRAP TABLE_11_01_07.csv')
df.to_csv(conversion, encoding='utf-8-sig', index=False)


df['study_id']="martin2008tubes"
df['test_condition']="test"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)

In [3]:
# df.columns
df.rename(columns={"subject": "ape",
    "species": "species_x",
    "sex": "sex_y",
    "tecnique_success": "technique_success",
    "filter_$": "filter_dollar"}, inplace=True)

In [4]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [5]:
# df.columns
df.rename(columns={"ape": "participant",
                   "age":"age_original",
                   "trap_side":'side_temp',
                   "side":'trap_side'}, inplace=True)

df['age_in_years'] = (df['age_original'])//12

In [6]:
df['session'].unique()
df.loc[df.session == 0., ['test_condition']] = 'pretest'

In [7]:
martin2008tubes_standardized=df[['study_id','participant','sex', 'age_original','age_in_years', 'species', 
       'session', 'trial','test_condition','condition', 
    #    'subj_number', 
       'trap_side', 
    #    'side', 
       'correct', 
       # 'latency',
       'right_hand', 'left_hand', 'hand_success',
       #   'right_side', 'left_side',
       'rake', 'push',
       #   'mixed_1', 'mixed_2',
           'twist', 'technique_success',
       'multiple_insertions']]
comp_out_path_stand = os.path.join(out_pathway, 'martin2008tubes_standardized.csv')
martin2008tubes_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =martin2008tubes_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
martin2008tubes_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'martin2008tubes_glossary.csv')
martin2008tubes_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
